# Notebook 11: Training Data Provenance Audit

Systematic audit of the `neoyipeng/financial_reasoning_aggregated` dataset.
Produces the provenance table, figures, and statistics for the paper.

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from collections import Counter
import re
import json
import warnings
warnings.filterwarnings("ignore")

SOURCE_NAMES = {
    3: "Earnings Calls (Narrative)",
    4: "Press Releases & News",
    5: "FinancialPhraseBank",
    8: "Earnings Calls (Q&A)",
    9: "Financial Tweets",
}

ds = load_dataset("neoyipeng/financial_reasoning_aggregated")
ds_sent = ds.filter(lambda x: x["task"] == "sentiment")

rows = []
for split in ds_sent:
    for r in ds_sent[split]:
        rows.append({
            "text": r["text"],
            "label": r["label"],
            "source": r["source"],
            "split": split,
            "prompt": r.get("prompt", ""),
            "word_count": len(str(r["text"]).split()),
            "char_count": len(str(r["text"])),
        })

df = pd.DataFrame(rows)
print(f"Total sentiment samples: {len(df):,}")
print(f"Sources: {sorted(df['source'].unique())}")
print(f"Splits: {df['split'].value_counts().to_dict()}")

## 2. Source Identification & Domain Verification

In [ ]:
for src, name in SOURCE_NAMES.items():
    sub = df[df["source"] == src]
    print(f"\nSource {src}: {name} (n={len(sub):,})")

    prompts = sub["prompt"].dropna().head(5)
    prompt_types = set()
    for p in prompts:
        if "earnings call" in p.lower(): prompt_types.add("earnings_call")
        elif "tweet" in p.lower(): prompt_types.add("tweet")
        elif "news" in p.lower(): prompt_types.add("news")
        elif "headline" in p.lower(): prompt_types.add("headline")
    print(f"  Prompt types: {prompt_types}")

    for _, row in sub.sample(3, random_state=42).iterrows():
        print(f"  [{row['label']}] {row['text'][:120]}...")

## 3. Annotation Method Audit

In [ ]:
print("=" * 70)
print("ANNOTATION METHOD ANALYSIS")
print("=" * 70)

for src in sorted(df["source"].unique()):
    sub = df[df["source"] == src]
    prompts = sub["prompt"].dropna()

    if src == 5:
        print(f"\nSource {src} (FPB): HUMAN-ANNOTATED")
        print("  16-24 finance professionals per sentence")
        print("  Agreement thresholds: 50%, 66%, 75%, 100%")
        continue

    has_cot = prompts.str.contains("Reason step by step", case=False).mean()
    has_classify = prompts.str.contains("Classify the sentiment", case=False).mean()

    print(f"\nSource {src} ({SOURCE_NAMES[src]}): LLM-GENERATED LABELS")
    print(f"  Prompts with 'Classify the sentiment': {has_classify:.0%}")
    print(f"  Prompts with CoT instruction: {has_cot:.0%}")
    print(f"  Sample prompt: {prompts.iloc[0][:200]}...")

print("\nKey: model trained on LLM labels, evaluated against human labels (FPB)")

## 4. Provenance Table (for paper)

In [ ]:
provenance = []
for src in sorted(df["source"].unique()):
    sub = df[df["source"] == src]
    train_sub = sub[sub["split"] == "train"]

    years = []
    for t in sub["text"]:
        found = re.findall(r"\b(20[12][0-9])\b", str(t))
        years.extend(found)
    time_period = f"{min(years)}-{max(years)}" if years else ""

    provenance.append({
        "Source ID": src,
        "Domain": SOURCE_NAMES[src],
        "N (total)": len(sub),
        "N (train)": len(train_sub),
        "Annotation": "Human" if src == 5 else "LLM",
        "NEG %": f"{(sub['label']=='NEGATIVE').mean()*100:.1f}",
        "NEU %": f"{(sub['label']=='NEUTRAL/MIXED').mean()*100:.1f}",
        "POS %": f"{(sub['label']=='POSITIVE').mean()*100:.1f}",
        "Med. Words": int(sub["word_count"].median()),
        "Time Period": time_period,
    })

prov_df = pd.DataFrame(provenance)
print(prov_df.to_string(index=False))

# Training set class balance
train_nf = df[(df["split"] == "train") & (df["source"] != 5)]
print(f"\nTraining set (excl FPB): {len(train_nf):,} samples")
for lbl in ["NEGATIVE", "NEUTRAL/MIXED", "POSITIVE"]:
    cnt = (train_nf["label"] == lbl).sum()
    print(f"  {lbl}: {cnt} ({cnt/len(train_nf)*100:.1f}%)")

## 5. Source 4: Mining Sub-Domain Analysis

In [ ]:
src4 = df[df["source"] == 4]
mining_keywords = r"TSX|hectare|drill|assay|mining|gold|copper|zinc|mineral|ore|exploration|deposit"
is_mining = src4["text"].str.contains(mining_keywords, case=False, regex=True)

print(f"Source 4 sub-domain breakdown:")
print(f"  Mining/Resources: {is_mining.sum()} ({is_mining.mean():.1%})")
print(f"  Other financial: {(~is_mining).sum()} ({(~is_mining).mean():.1%})")

for label, subset in [("Mining", src4[is_mining]), ("Non-mining", src4[~is_mining])]:
    print(f"\n  {label} label dist:")
    for lbl in ["NEGATIVE", "NEUTRAL/MIXED", "POSITIVE"]:
        print(f"    {lbl}: {(subset['label']==lbl).mean()*100:.1f}%")

## 6. Source 8: Truncation Analysis

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

src8 = df[df["source"] == 8]
token_counts = []
for text in src8["text"]:
    tokens = tokenizer(text, truncation=False)["input_ids"]
    token_counts.append(len(tokens))

token_counts = np.array(token_counts)
truncated_512 = (token_counts > 512).sum()
truncated_256 = (token_counts > 256).sum()

print(f"Source 8 tokenization analysis (ModernBERT tokenizer):")
print(f"  Total samples: {len(token_counts)}")
print(f"  Token count: min={token_counts.min()}, median={int(np.median(token_counts))}, "
      f"mean={int(token_counts.mean())}, max={token_counts.max()}")
print(f"  Truncated at 512 tokens: {truncated_512} ({truncated_512/len(token_counts):.1%})")
print(f"  Truncated at 256 tokens: {truncated_256} ({truncated_256/len(token_counts):.1%})")

## 7. LLM vs Human Annotation Bias

In [ ]:
fpb = df[df["source"] == 5]
fpb_dist = fpb["label"].value_counts(normalize=True).sort_index()

print("LLM vs HUMAN ANNOTATION BIAS")
print("=" * 60)
for src in [3, 4, 8, 9]:
    sub = df[df["source"] == src]
    sub_dist = sub["label"].value_counts(normalize=True).sort_index()
    print(f"\nSource {src} ({SOURCE_NAMES[src]}) vs FPB:")
    for lbl in ["NEGATIVE", "NEUTRAL/MIXED", "POSITIVE"]:
        src_pct = sub_dist.get(lbl, 0) * 100
        fpb_pct = fpb_dist.get(lbl, 0) * 100
        delta = src_pct - fpb_pct
        print(f"  {lbl}: {src_pct:.1f}% (vs FPB {fpb_pct:.1f}%, delta={delta:+.1f}pp)")

## 8. Duplicate Checks

In [ ]:
print("Intra-source duplicates:")
for src in sorted(df["source"].unique()):
    sub = df[df["source"] == src]
    dupes = sub[sub.duplicated(subset="text", keep=False)]
    n_unique = len(dupes) // 2 if len(dupes) > 0 else 0
    print(f"  Source {src}: {n_unique} duplicated texts")

text_sources = df.groupby("text")["source"].nunique()
multi = (text_sources > 1).sum()
print(f"\nCross-source duplicates: {multi}")

## 9. Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

source_order = [9, 5, 3, 4, 8]
source_labels = [f"{SOURCE_NAMES[s]}\n(n={len(df[df['source']==s]):,})" for s in source_order]
data_for_box = [df[df["source"] == s]["word_count"].clip(upper=300) for s in source_order]

bp = axes[0].boxplot(data_for_box, labels=source_labels, patch_artist=True, showfliers=False)
colors = ["#FF9800", "#9C27B0", "#2196F3", "#4CAF50", "#F44336"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[0].set_ylabel("Word Count")
axes[0].set_title("(a) Text Length Distribution by Source")
axes[0].tick_params(axis="x", rotation=20, labelsize=8)

label_colors_map = {"NEGATIVE": "#F44336", "NEUTRAL/MIXED": "#9E9E9E", "POSITIVE": "#4CAF50"}
x_labels = [SOURCE_NAMES[s] for s in source_order]
for i, src in enumerate(source_order):
    sub = df[df["source"] == src]
    total = len(sub)
    bottom = 0
    for label in ["POSITIVE", "NEUTRAL/MIXED", "NEGATIVE"]:
        pct = (sub["label"] == label).sum() / total * 100
        bar_label = label if i == 0 else ""
        axes[1].bar(x_labels[i], pct, bottom=bottom,
                   color=label_colors_map[label], label=bar_label)
        bottom += pct

axes[1].set_ylabel("Percentage (%)")
axes[1].set_title("(b) Label Distribution by Source")
axes[1].legend(loc="upper right", fontsize=8)
axes[1].tick_params(axis="x", rotation=20, labelsize=8)

plt.tight_layout()
plt.savefig("../results/data_provenance_figure.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(token_counts, bins=50, alpha=0.7, color="#2196F3", edgecolor="black")
ax.axvline(512, color="red", linestyle="--", linewidth=2, label="512-token limit")
ax.set_xlabel("Token Count")
ax.set_ylabel("Frequency")
ax.set_title("Source 8 (Earnings Call Q&A) Token Length Distribution")
ax.legend()
plt.tight_layout()
plt.savefig("../results/source8_truncation.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Export & Summary

In [ ]:
audit_results = {
    "dataset": "neoyipeng/financial_reasoning_aggregated",
    "total_sentiment_samples": len(df),
    "training_samples_excl_fpb": len(train_nf),
    "sources": {},
    "truncation_analysis": {
        "source_8_total": len(token_counts),
        "source_8_truncated_at_512": int(truncated_512),
        "source_8_truncated_pct": round(truncated_512 / len(token_counts), 3),
        "source_8_median_tokens": int(np.median(token_counts)),
        "source_8_max_tokens": int(token_counts.max()),
    },
    "source_4_mining_pct": round(is_mining.mean(), 3),
}

for src in sorted(df["source"].unique()):
    sub = df[df["source"] == src]
    audit_results["sources"][str(src)] = {
        "name": SOURCE_NAMES[src],
        "n_total": len(sub),
        "n_train": len(sub[sub["split"] == "train"]),
        "annotation_method": "human" if src == 5 else "llm",
        "label_distribution": {
            "NEGATIVE": round((sub["label"] == "NEGATIVE").mean(), 3),
            "NEUTRAL/MIXED": round((sub["label"] == "NEUTRAL/MIXED").mean(), 3),
            "POSITIVE": round((sub["label"] == "POSITIVE").mean(), 3),
        },
        "text_length": {
            "median_words": int(sub["word_count"].median()),
            "mean_words": round(sub["word_count"].mean(), 1),
            "max_words": int(sub["word_count"].max()),
        },
    }

with open("../results/data_provenance_audit.json", "w") as f:
    json.dump(audit_results, f, indent=2)

print("Saved to results/data_provenance_audit.json")
print()
print("KEY FINDINGS:")
print("1. All non-FPB labels are LLM-generated")
print(f"2. Source 4 is {is_mining.mean():.0%} Canadian mining press releases")
print(f"3. Source 8: {truncated_512/len(token_counts):.0%} of samples truncated at 512 tokens")
print(f"4. Class imbalance: Source 4 has 3.5% NEG vs Source 9 at 13.9%")
print(f"5. Training NEG rate (10.2%) vs FPB NEG rate (12.5%)")